In [160]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd
import numpy as np
import sys
# sys.stdout = open('file.txt', 'w')

In [161]:
spark = SparkSession.builder.appName("seer_pre_prop").getOrCreate()

In [162]:
dic_dados = pd.read_csv('./SEER_1975_2016_CUSTOM_TEXTDATA/carga_inicial.csv', sep=';')

In [163]:
df = spark.read.csv('./SEER_1975_2016_CUSTOM_TEXTDATA/incidence/*', pathGlobFilter='*.TXT')

In [164]:
df = df.withColumnRenamed('_c0', 'linha')

In [165]:
colunas = []
for i in range(dic_dados.shape[0]):
    linha = dic_dados.iloc[i, :]
    nome_coluna = linha[0]
    tipo_coluna = linha[1]
    pos_coluna = int(linha[3])
    if i > 0:
        pos_coluna += 1
    tam_coluna = int(linha[4])
    colunas.append(F.trim(F.substring(df['linha'], pos_coluna, tam_coluna)).cast(tipo_coluna).alias(nome_coluna))

In [166]:
df = df.select(colunas).replace('', None)

In [167]:
for i in range(dic_dados.shape[0]):
    linha = dic_dados.iloc[i, :]
    nome_coluna = linha[0]
    try:
        rep_nulo = int(linha[-4])
    except ValueError:
        continue
    df = df.replace(rep_nulo, None, subset=nome_coluna)

In [168]:
tipo_tumor = F.split(F.split(F.substring(F.input_file_name(), -68, 100), '/').getItem(3), '.TXT').getItem(0)
base_ano = F.split(F.substring(F.input_file_name(), -68, 100), '/').getItem(2)

In [169]:
df = df.withColumn('TIPO_TUMOR', tipo_tumor).withColumn('BASE_ANO', base_ano)

In [ ]:
df.select('TIPO_TUMOR').groupby('TIPO_TUMOR').count().show()

In [ ]:
df.select('BASE_ANO').groupby('BASE_ANO').count().show()

In [44]:
TAM_TOTAL = df.count()

In [ ]:
# df.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) / TAM_TOTAL for c in df.columns]).show()

In [20]:
coluna = 'PUBCSNUM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT PUBCSNUM)|(count(CASE WHEN (isnan(PUBCSNUM) OR (PUBCSNUM IS NULL)) THEN PUBCSNUM END) AS `PUBCSNUM` / 10450709)|PUBCSNUM|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                 9157072|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-----+
|PUBCSNUM|count|
+--------+-----+
|23482728|   47|
| 7283969|   24|
|11770208|   23|
| 1221181|   22|
|50436082|   22|
|58413108|   22|
|69999427|   19|
|45497996|   19|
|23484808|   19|
|35055767|   18|
| 1181914|   18|
|74848066|   16|
|43813542|   16|
| 7910155|   15|
|45555794|   15|
|48

In [21]:
coluna = 'REG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-------------------+---------------------------------------------------------------------------------+---+
|count(DISTINCT REG)|(count(CASE WHEN (isnan(REG) OR (REG IS NULL)) THEN REG END) AS `REG` / 10450709)|REG|
+-------------------+---------------------------------------------------------------------------------+---+
|                 18|                                                                              0.0|  0|
+-------------------+---------------------------------------------------------------------------------+---+

+----+-------+
| REG|  count|
+----+-------+
|1541|1641033|
|1535|1004075|
|1544| 933605|
|1520| 883838|
|1525| 804893|
|1501| 804050|
|1502| 786931|
|1522| 660766|
|1547| 533643|
|1542| 459716|
|1543| 426410|
|1527| 425972|
|1526| 288504|
|1523| 286268|
|1531| 264366|
|1521| 219251|
|1537|  17708|
|1529|   9680|
+----+-------+



In [22]:
coluna = 'MAR_STAT'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT MAR_STAT)|(count(CASE WHEN (isnan(MAR_STAT) OR (MAR_STAT IS NULL)) THEN MAR_STAT END) AS `MAR_STAT` / 10450709)|MAR_STAT|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       6|                                                                                  0.07786371240458423|  813731|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|MAR_STAT|  count|
+--------+-------+
|       2|5692089|
|       5|1562666|
|       1|1409378|
|       4| 850064|
|    null| 813731|
|       3| 113400|
|       6|   9381|
+--------+-------+



In [24]:
coluna = 'RACE1V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+------+
|count(DISTINCT RACE1V)|(count(CASE WHEN (isnan(RACE1V) OR (RACE1V IS NULL)) THEN RACE1V END) AS `RACE1V` / 10450709)|RACE1V|
+----------------------+---------------------------------------------------------------------------------------------+------+
|                    29|                                                                         0.009225115731382435| 96409|
+----------------------+---------------------------------------------------------------------------------------------+------+

+------+-------+
|RACE1V|  count|
+------+-------+
|     1|8661749|
|     2|1018928|
|     6| 134367|
|     4| 125129|
|     5| 104583|
|  null|  96409|
|    96|  55010|
|     3|  51658|
|     8|  40833|
|    10|  39869|
|     7|  36747|
|    16|  21465|
|    98|  17303|
|    15|  16312|
|    27|   5259|
|    97|   4464|
|    13|   4289|
|    11|   3734|
|    14|   3

In [25]:
coluna = 'NHIADE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+--------+
|count(DISTINCT NHIADE)|(count(CASE WHEN (isnan(NHIADE) OR (NHIADE IS NULL)) THEN NHIADE END) AS `NHIADE` / 10450709)|  NHIADE|
+----------------------+---------------------------------------------------------------------------------------------+--------+
|                     0|                                                                                          1.0|10450709|
+----------------------+---------------------------------------------------------------------------------------------+--------+

+------+--------+
|NHIADE|   count|
+------+--------+
|  null|10450709|
+------+--------+



In [26]:
coluna = 'SEX'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-------------------+---------------------------------------------------------------------------------+---+
|count(DISTINCT SEX)|(count(CASE WHEN (isnan(SEX) OR (SEX IS NULL)) THEN SEX END) AS `SEX` / 10450709)|SEX|
+-------------------+---------------------------------------------------------------------------------+---+
|                  2|                                                                              0.0|  0|
+-------------------+---------------------------------------------------------------------------------+---+

+---+-------+
|SEX|  count|
+---+-------+
|  2|5272895|
|  1|5177814|
+---+-------+



In [27]:
coluna = 'AGE_DX'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+------+
|count(DISTINCT AGE_DX)|(count(CASE WHEN (isnan(AGE_DX) OR (AGE_DX IS NULL)) THEN AGE_DX END) AS `AGE_DX` / 10450709)|AGE_DX|
+----------------------+---------------------------------------------------------------------------------------------+------+
|                   118|                                                                         1.351104504010206...|  1412|
+----------------------+---------------------------------------------------------------------------------------------+------+

+------+------+
|AGE_DX| count|
+------+------+
|    65|289220|
|    67|283213|
|    68|283119|
|    66|282657|
|    69|280909|
|    70|275635|
|    71|274854|
|    72|273266|
|    73|270399|
|    64|264303|
|    74|263523|
|    63|261394|
|    75|256652|
|    62|255884|
|    76|250479|
|    61|249235|
|    60|242156|
|    77|241148|
|    59|234434|
|    78|231700|
+

In [28]:
coluna = 'YR_BRTH'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT YR_BRTH)|(count(CASE WHEN (isnan(YR_BRTH) OR (YR_BRTH IS NULL)) THEN YR_BRTH END) AS `YR_BRTH` / 10450709)|YR_BRTH|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    148|                                                                             1.351104504010206...|   1412|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+------+
|YR_BRTH| count|
+-------+------+
|   1947|233924|
|   1942|225599|
|   1943|224573|
|   1946|221817|
|   1930|214409|
|   1948|210656|
|   1938|209947|
|   1927|209416|
|   1941|208961|
|   1944|207924|
|   1928|207820|
|   1929|207460|
|   1932|207374|
|   1931|206509|
|   1937|206425|
|   1940|205817|
|   1934|205768|

In [29]:
coluna = 'SEQ_NUM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT SEQ_NUM)|(count(CASE WHEN (isnan(SEQ_NUM) OR (SEQ_NUM IS NULL)) THEN SEQ_NUM END) AS `SEQ_NUM` / 10450709)|SEQ_NUM|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                     70|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|SEQ_NUM|  count|
+-------+-------+
|      0|7423004|
|      2|1445978|
|      1|1086736|
|      3| 245906|
|     60| 179252|
|      4|  44747|
|      5|   9785|
|     62|   4722|
|     61|   3963|
|      6|   2913|
|      7|   1059|
|     99|    548|
|      8|    530|
|     63|    397|
|      9|    286|
|     10|    18

In [30]:
coluna = 'MDXRECMP'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT MDXRECMP)|(count(CASE WHEN (isnan(MDXRECMP) OR (MDXRECMP IS NULL)) THEN MDXRECMP END) AS `MDXRECMP` / 10450709)|MDXRECMP|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      12|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+------+
|MDXRECMP| count|
+--------+------+
|       7|933042|
|       3|904383|
|       1|899237|
|      10|894377|
|       6|889575|
|       8|885548|
|       5|882440|
|       4|875371|
|       9|842430|
|      11|824972|
|       2|810172|
|      12|809162|
+--------+------+



In [31]:
coluna = 'YEAR_DX'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT YEAR_DX)|(count(CASE WHEN (isnan(YEAR_DX) OR (YEAR_DX IS NULL)) THEN YEAR_DX END) AS `YEAR_DX` / 10450709)|YEAR_DX|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                     42|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+------+
|YEAR_DX| count|
+-------+------+
|   2015|501490|
|   2016|497931|
|   2014|491152|
|   2013|481189|
|   2012|475497|
|   2011|472833|
|   2010|466263|
|   2009|465204|
|   2008|455330|
|   2007|448793|
|   2006|431974|
|   2005|421087|
|   2004|415881|
|   2002|399994|
|   2003|396136|
|   2001|395196|
|   2000|380917|

In [32]:
coluna = 'PRIMSITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT PRIMSITE)|(count(CASE WHEN (isnan(PRIMSITE) OR (PRIMSITE IS NULL)) THEN PRIMSITE END) AS `PRIMSITE` / 10450709)|PRIMSITE|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                     330|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|PRIMSITE|  count|
+--------+-------+
|    C619|1310121|
|    C341| 594111|
|    C504| 554600|
|    C421| 507597|
|    C508| 351868|
|    C509| 305085|
|    C343| 288853|
|    C649| 260922|
|    C349| 241770|
|    C187| 224526|
|    C541| 221277|
|    C209| 216286|
|    C739| 212938|
|    

In [45]:
coluna = 'LATERAL'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT LATERAL)|(count(CASE WHEN (isnan(LATERAL) OR (LATERAL IS NULL)) THEN LATERAL END) AS `LATERAL` / 10450709)|LATERAL|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                      7|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|LATERAL|  count|
+-------+-------+
|      0|5879066|
|      1|2198958|
|      2|2052789|
|      9| 196652|
|      4|  81992|
|      3|  23728|
|      5|  17524|
+-------+-------+



In [46]:
coluna = 'HISTO2V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT HISTO2V)|(count(CASE WHEN (isnan(HISTO2V) OR (HISTO2V IS NULL)) THEN HISTO2V END) AS `HISTO2V` / 10450709)|HISTO2V|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    577|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|HISTO2V|  count|
+-------+-------+
|   8140|2846959|
|   8500|1203683|
|   8070| 644963|
|   8010| 456301|
|   8720| 371140|
|   8130| 297774|
|   8000| 247285|
|   8520| 154556|
|   9680| 151771|
|   8380| 150929|
|   8041| 147388|
|   8743| 146822|
|   8260| 135829|
|   8120| 133194|
|   8480| 125117|
|   8310| 11814

In [47]:
coluna = 'BEHO2V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+------+
|count(DISTINCT BEHO2V)|(count(CASE WHEN (isnan(BEHO2V) OR (BEHO2V IS NULL)) THEN BEHO2V END) AS `BEHO2V` / 10450709)|BEHO2V|
+----------------------+---------------------------------------------------------------------------------------------+------+
|                     4|                                                                                          0.0|     0|
+----------------------+---------------------------------------------------------------------------------------------+------+

+------+-------+
|BEHO2V|  count|
+------+-------+
|     3|9435645|
|     2| 718107|
|     0| 176011|
|     1| 120946|
+------+-------+



In [48]:
coluna = 'HISTO3V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT HISTO3V)|(count(CASE WHEN (isnan(HISTO3V) OR (HISTO3V IS NULL)) THEN HISTO3V END) AS `HISTO3V` / 10450709)|HISTO3V|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    707|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|HISTO3V|  count|
+-------+-------+
|   8140|2783010|
|   8500|1107290|
|   8070| 637500|
|   8720| 370758|
|   8010| 354059|
|   8130| 298015|
|   8000| 243830|
|   8520| 150726|
|   8380| 150634|
|   8041| 147514|
|   8743| 147041|
|   8260| 136280|
|   8120| 132510|
|   9680| 127210|
|   8480| 124814|
|   8310| 11831

In [49]:
coluna = 'BEHO3V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+------+
|count(DISTINCT BEHO3V)|(count(CASE WHEN (isnan(BEHO3V) OR (BEHO3V IS NULL)) THEN BEHO3V END) AS `BEHO3V` / 10450709)|BEHO3V|
+----------------------+---------------------------------------------------------------------------------------------+------+
|                     4|                                                                                          0.0|     0|
+----------------------+---------------------------------------------------------------------------------------------+------+

+------+-------+
|BEHO3V|  count|
+------+-------+
|     3|9537684|
|     2| 718109|
|     0| 172094|
|     1|  22822|
+------+-------+



In [50]:
coluna = 'GRADE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+---------------------+-----------------------------------------------------------------------------------------+-------+
|count(DISTINCT GRADE)|(count(CASE WHEN (isnan(GRADE) OR (GRADE IS NULL)) THEN GRADE END) AS `GRADE` / 10450709)|  GRADE|
+---------------------+-----------------------------------------------------------------------------------------+-------+
|                    8|                                                                      0.40466431511967277|4229029|
+---------------------+-----------------------------------------------------------------------------------------+-------+

+-----+-------+
|GRADE|  count|
+-----+-------+
| null|4229029|
|    2|2534728|
|    3|1936564|
|    1| 919666|
|    4| 405657|
|    6| 389097|
|    5|  33680|
|    8|   1318|
|    7|    970|
+-----+-------+



In [51]:
coluna = 'DX_CONF'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT DX_CONF)|(count(CASE WHEN (isnan(DX_CONF) OR (DX_CONF IS NULL)) THEN DX_CONF END) AS `DX_CONF` / 10450709)|DX_CONF|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                      8|                                                                              0.01779879240729026| 186010|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|DX_CONF|  count|
+-------+-------+
|      1|9349206|
|      2| 352887|
|      7| 348158|
|   null| 186010|
|      8|  87086|
|      3|  66795|
|      5|  31237|
|      6|  20582|
|      4|   8748|
+-------+-------+



In [52]:
coluna = 'REPT_SRC'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT REPT_SRC)|(count(CASE WHEN (isnan(REPT_SRC) OR (REPT_SRC IS NULL)) THEN REPT_SRC END) AS `REPT_SRC` / 10450709)|REPT_SRC|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       8|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|REPT_SRC|  count|
+--------+-------+
|       1|9525364|
|       4| 382610|
|       3| 270889|
|       7|  97732|
|       8|  73672|
|       2|  65275|
|       6|  22145|
|       5|  13022|
+--------+-------+



In [53]:
coluna = 'EOD10_SZ'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD10_SZ)|(count(CASE WHEN (isnan(EOD10_SZ) OR (EOD10_SZ IS NULL)) THEN EOD10_SZ END) AS `EOD10_SZ` / 10450709)|EOD10_SZ|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                     718|                                                                                    0.842053682673587| 8800058|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|EOD10_SZ|  count|
+--------+-------+
|    null|8800058|
|      30| 113354|
|      40| 102260|
|      20| 102101|
|      50|  91444|
|      15|  79766|
|      25|  76731|
|       1|  66744|
|      10|  63279|
|      60|  62818|
|      35|  57854|
|      70|  42807|
|      45|  42232|
|    

In [54]:
coluna = 'EOD10_EX'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD10_EX)|(count(CASE WHEN (isnan(EOD10_EX) OR (EOD10_EX IS NULL)) THEN EOD10_EX END) AS `EOD10_EX` / 10450709)|EOD10_EX|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      72|                                                                                   0.7085717342239651| 7405077|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|EOD10_EX|  count|
+--------+-------+
|    null|7405077|
|      10| 577395|
|      85| 368118|
|      30| 217912|
|      20| 203342|
|       0| 197020|
|      80| 193008|
|      40| 164452|
|      15| 156963|
|      11| 110183|
|      45|  82665|
|      50|  68536|
|      60|  68394|
|    

In [55]:
coluna = 'EOD10_PE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD10_PE)|(count(CASE WHEN (isnan(EOD10_PE) OR (EOD10_PE IS NULL)) THEN EOD10_PE END) AS `EOD10_PE` / 10450709)|EOD10_PE|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      28|                                                                                   0.9883548570723766|10329009|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+--------+
|EOD10_PE|   count|
+--------+--------+
|    null|10329009|
|      23|   24536|
|      90|   23249|
|      20|   11188|
|      34|   10949|
|      40|    9759|
|      32|    9703|
|      30|    6622|
|      45|    4938|
|      42|    4628|
|      48|    4070|
|      41|    3229|
|      33

In [56]:
coluna = 'EOD10_ND'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD10_ND)|(count(CASE WHEN (isnan(EOD10_ND) OR (EOD10_ND IS NULL)) THEN EOD10_ND END) AS `EOD10_ND` / 10450709)|EOD10_ND|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       9|                                                                                   0.7794530495490785| 8145837|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|EOD10_ND|  count|
+--------+-------+
|    null|8145837|
|       0|1692263|
|       1| 163993|
|       2| 160150|
|       6|  99515|
|       3|  79728|
|       5|  47354|
|       7|  28406|
|       4|  23553|
|       8|   9910|
+--------+-------+



In [57]:
coluna = 'EOD10_PN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD10_PN)|(count(CASE WHEN (isnan(EOD10_PN) OR (EOD10_PN IS NULL)) THEN EOD10_PN END) AS `EOD10_PN` / 10450709)|EOD10_PN|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      93|                                                                                   0.2547304685261067| 2662114|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|EOD10_PN|  count|
+--------+-------+
|      98|4717119|
|    null|2662114|
|       0|2040597|
|       1| 357351|
|       2| 163618|
|       3|  95318|
|      95|  66032|
|       4|  65405|
|      97|  47574|
|       5|  46975|
|       6|  34787|
|       7|  27156|
|       8|  21616|
|    

In [58]:
coluna = 'EOD10_NE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD10_NE)|(count(CASE WHEN (isnan(EOD10_NE) OR (EOD10_NE IS NULL)) THEN EOD10_NE END) AS `EOD10_NE` / 10450709)|EOD10_NE|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      98|                                                                                  0.25442072877543526| 2658877|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|EOD10_NE|  count|
+--------+-------+
|       0|4717119|
|    null|2658877|
|       1| 343497|
|       2| 289061|
|       3| 207555|
|       4| 161282|
|       5| 133135|
|       6| 119348|
|      98| 118407|
|       7| 109769|
|       8| 105287|
|      12| 103437|
|      10| 102609|
|    

In [59]:
coluna = 'EOD13'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+---------------------+-----------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD13)|(count(CASE WHEN (isnan(EOD13) OR (EOD13 IS NULL)) THEN EOD13 END) AS `EOD13` / 10450709)|   EOD13|
+---------------------+-----------------------------------------------------------------------------------------+--------+
|               108636|                                                                       0.9653340266196293|10088425|
+---------------------+-----------------------------------------------------------------------------------------+--------+

+-------------+--------+
|        EOD13|   count|
+-------------+--------+
|         null|10088425|
|-----------00|    5541|
|0--01--80-000|    4537|
|----600-0-000|    3390|
|0--01--20-000|    3272|
|0--31--20-000|    1947|
|0--31--80-000|    1886|
|----200-0-000|    1571|
|0--0---80-000|    1546|
|----100-0-000|    1342|
|-----00-0-000|    1109|
|---410000-000|    1064|
|0--01--5

In [60]:
coluna = 'EOD2'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+--------------------+-------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD2)|(count(CASE WHEN (isnan(EOD2) OR (EOD2 IS NULL)) THEN EOD2 END) AS `EOD2` / 10450709)|    EOD2|
+--------------------+-------------------------------------------------------------------------------------+--------+
|                 125|                                                                   0.9768669283586405|10208952|
+--------------------+-------------------------------------------------------------------------------------+--------+

+----+--------+
|EOD2|   count|
+----+--------+
|null|10208952|
|  --|   44788|
|  4-|   42185|
|  &-|   38584|
|  &1|   23587|
|  10|   13085|
|  0-|    5223|
|  70|    4386|
|  5-|    4049|
|  &6|    3966|
|  11|    2759|
|  71|    2719|
|  19|    2494|
|  79|    2228|
|  59|    2210|
|  12|    2185|
|  &2|    1899|
|  &3|    1846|
|  90|    1844|
|  15|    1726|
+----+--------+
only showing top 20 rows


In [61]:
coluna = 'EOD4'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+--------------------+-------------------------------------------------------------------------------------+-------+
|count(DISTINCT EOD4)|(count(CASE WHEN (isnan(EOD4) OR (EOD4 IS NULL)) THEN EOD4 END) AS `EOD4` / 10450709)|   EOD4|
+--------------------+-------------------------------------------------------------------------------------+-------+
|                4300|                                                                     0.95568970488031|9987635|
+--------------------+-------------------------------------------------------------------------------------+-------+

+----+-------+
|EOD4|  count|
+----+-------+
|null|9987635|
|9989|  46204|
|9999|  45735|
|9910|  22480|
|9939|  14987|
|9930|  13369|
|9919|  11067|
|9900|   9335|
|9920|   8842|
|9980|   6995|
|2010|   5764|
|9979|   5525|
| 110|   5364|
|1510|   5203|
|9929|   5052|
| 100|   4671|
|9981|   4342|
|9940|   4227|
|1010|   3961|
|3010|   3957|
+----+-------+
only showing top 20 rows



In [62]:
coluna = 'EOD_CODE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT EOD_CODE)|(count(CASE WHEN (isnan(EOD_CODE) OR (EOD_CODE IS NULL)) THEN EOD_CODE END) AS `EOD_CODE` / 10450709)|EOD_CODE|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       5|                                                                                   0.5764799306917837| 6024624|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|EOD_CODE|  count|
+--------+-------+
|    null|6024624|
|       4|3358970|
|       3| 463074|
|       2| 362284|
|       1| 137958|
|       0| 103799|
+--------+-------+



In [63]:
coluna = 'TUMOR_1V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT TUMOR_1V)|(count(CASE WHEN (isnan(TUMOR_1V) OR (TUMOR_1V IS NULL)) THEN TUMOR_1V END) AS `TUMOR_1V` / 10450709)|TUMOR_1V|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       8|                                                                                    0.949222966594898| 9920053|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|TUMOR_1V|  count|
+--------+-------+
|    null|9920053|
|       1| 282858|
|       0| 142576|
|       2|  87171|
|       8|  13744|
|       3|   2259|
|       4|   1535|
|       5|    339|
|       6|    174|
+--------+-------+



In [64]:
coluna = 'TUMOR_2V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT TUMOR_2V)|(count(CASE WHEN (isnan(TUMOR_2V) OR (TUMOR_2V IS NULL)) THEN TUMOR_2V END) AS `TUMOR_2V` / 10450709)|TUMOR_2V|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       8|                                                                                   0.9578372146808413|10010078|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+--------+
|TUMOR_2V|   count|
+--------+--------+
|    null|10010078|
|       1|  231394|
|       2|  118877|
|       0|   71039|
|       8|   13876|
|       3|    3293|
|       4|    1704|
|       5|     294|
|       6|     154|
+--------+--------+



In [65]:
coluna = 'TUMOR_3V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT TUMOR_3V)|(count(CASE WHEN (isnan(TUMOR_3V) OR (TUMOR_3V IS NULL)) THEN TUMOR_3V END) AS `TUMOR_3V` / 10450709)|TUMOR_3V|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       6|                                                                                   0.9996323694401978|10446867|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+--------+
|TUMOR_3V|   count|
+--------+--------+
|    null|10446867|
|       0|    1598|
|       2|     985|
|       4|     577|
|       8|     256|
|       5|     245|
|       6|     181|
+--------+--------+



In [66]:
coluna = 'CSTUMSIZ'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CSTUMSIZ)|(count(CASE WHEN (isnan(CSTUMSIZ) OR (CSTUMSIZ IS NULL)) THEN CSTUMSIZ END) AS `CSTUMSIZ` / 10450709)|CSTUMSIZ|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                     621|                                                                                   0.6530277515142753| 6824603|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CSTUMSIZ|  count|
+--------+-------+
|    null|6824603|
|     988| 540702|
|      30| 143885|
|      20| 139496|
|      15| 129006|
|      40| 118867|
|      25| 116273|
|      10| 109498|
|      50| 105656|
|      35|  86287|
|      12|  75476|
|      60|  73049|
|      45|  63267|
|    

In [67]:
coluna = 'CSEXTEN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CSEXTEN)|(count(CASE WHEN (isnan(CSEXTEN) OR (CSEXTEN IS NULL)) THEN CSEXTEN END) AS `CSEXTEN` / 10450709)|CSEXTEN|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    248|                                                                               0.5126800487890343|5357870|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CSEXTEN|  count|
+-------+-------+
|   null|5357870|
|    100|1233725|
|    150| 408813|
|      0| 407205|
|    300| 381514|
|    800| 356164|
|    400| 255607|
|    200| 244574|
|     50| 195597|
|    450| 112326|
|    110|  95413|
|    988|  86337|
|     10|  83758|
|    500|  76833|
|    600|  76479|
|    120|  6985

In [68]:
coluna = 'CSLYMPHN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CSLYMPHN)|(count(CASE WHEN (isnan(CSLYMPHN) OR (CSLYMPHN IS NULL)) THEN CSLYMPHN END) AS `CSLYMPHN` / 10450709)|CSLYMPHN|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                     120|                                                                                   0.5104708206878595| 5334782|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CSLYMPHN|  count|
+--------+-------+
|    null|5334782|
|       0|3256170|
|     988| 813087|
|     200| 243636|
|     100| 223175|
|     250| 106610|
|     600| 100060|
|     300|  58386|
|     110|  53744|
|     987|  28676|
|     150|  21569|
|     800|  19484|
|     120|  17130|
|    

In [69]:
coluna = 'CSMETSDX'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CSMETSDX)|(count(CASE WHEN (isnan(CSMETSDX) OR (CSMETSDX IS NULL)) THEN CSMETSDX END) AS `CSMETSDX` / 10450709)|CSMETSDX|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      54|                                                                                   0.4985833975474774| 5210550|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CSMETSDX|  count|
+--------+-------+
|    null|5210550|
|       0|3940366|
|      98| 622216|
|      40| 368078|
|      50|  43930|
|      30|  27307|
|      23|  25912|
|      60|  25734|
|      26|  20328|
|      44|  19291|
|      42|  15938|
|      10|  15618|
|      15|  14373|
|    

In [70]:
coluna = 'CS1SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CS1SITE)|(count(CASE WHEN (isnan(CS1SITE) OR (CS1SITE IS NULL)) THEN CS1SITE END) AS `CS1SITE` / 10450709)|CS1SITE|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    992|                                                                                0.644052666665965|6730807|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CS1SITE|  count|
+-------+-------+
|   null|6730807|
|     10|1138490|
|     20| 506732|
|      0| 460361|
|    988| 344336|
|    998| 239504|
|     40|  65994|
|     30|  49864|
|    997|  32241|
|    980|  27819|
|     50|  21436|
|    100|  19947|
|     45|  17083|
|     60|  16167|
|    400|  13915|
|     55|  1316

In [71]:
coluna = 'CS2SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CS2SITE)|(count(CASE WHEN (isnan(CS2SITE) OR (CS2SITE IS NULL)) THEN CS2SITE END) AS `CS2SITE` / 10450709)|CS2SITE|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    274|                                                                               0.7294989268192235|7623781|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CS2SITE|  count|
+-------+-------+
|   null|7623781|
|      0|1017079|
|     10| 715091|
|    988| 386907|
|    998| 322039|
|     20| 266380|
|    987|  24793|
|    400|  21450|
|      1|  17663|
|     40|  14126|
|    100|  10820|
|    997|   9195|
|     30|   6211|
|    200|   3875|
|      5|   2408|
|    110|   171

In [72]:
coluna = 'CS3SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CS3SITE)|(count(CASE WHEN (isnan(CS3SITE) OR (CS3SITE IS NULL)) THEN CS3SITE END) AS `CS3SITE` / 10450709)|CS3SITE|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    263|                                                                               0.7520926092191449|7859901|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CS3SITE|  count|
+-------+-------+
|   null|7859901|
|      0| 963175|
|    970| 448419|
|     98| 294557|
|      5| 229640|
|    230| 107904|
|      1|  98155|
|     10|  40369|
|      2|  38414|
|    960|  36594|
|    988|  35168|
|    210|  22622|
|      3|  21443|
|     11|  15696|
|    485|  15486|
|      4|  1404

In [73]:
coluna = 'CS4SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CS4SITE)|(count(CASE WHEN (isnan(CS4SITE) OR (CS4SITE IS NULL)) THEN CS4SITE END) AS `CS4SITE` / 10450709)|CS4SITE|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    118|                                                                               0.7655091152188813|8000113|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CS4SITE|  count|
+-------+-------+
|   null|8000113|
|      0| 937279|
|    988| 298973|
|    987| 275786|
|      1| 204879|
|    998| 177427|
|    550| 158938|
|    150|  82027|
|    250|  66328|
|     10|  43692|
|    110|  35855|
|    450|  20742|
|      2|  14167|
|    350|  12258|
|    990|   9821|
|    220|   865

In [74]:
coluna = 'CS5SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CS5SITE)|(count(CASE WHEN (isnan(CS5SITE) OR (CS5SITE IS NULL)) THEN CS5SITE END) AS `CS5SITE` / 10450709)|CS5SITE|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                     71|                                                                               0.8116925846849242|8482763|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CS5SITE|  count|
+-------+-------+
|   null|8482763|
|      0| 882427|
|    988| 393104|
|    987| 275448|
|     33| 137298|
|     34|  83751|
|     98|  63203|
|      1|  33989|
|     43|  32397|
|     44|  20399|
|     45|  13946|
|     99|   5216|
|     54|   4958|
|     35|   3265|
|     55|   2402|
|     32|   226

In [75]:
coluna = 'CS6SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT CS6SITE)|(count(CASE WHEN (isnan(CS6SITE) OR (CS6SITE IS NULL)) THEN CS6SITE END) AS `CS6SITE` / 10450709)|CS6SITE|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                     88|                                                                               0.8054909958740598|8417952|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|CS6SITE|  count|
+-------+-------+
|   null|8417952|
|      0| 489058|
|    988| 395799|
|     20| 363694|
|     10| 203390|
|      6| 141167|
|      7| 119132|
|     30|  89060|
|    987|  59203|
|     50|  45047|
|      8|  26608|
|     40|  24715|
|      9|  20432|
|    998|  14818|
|     60|   9124|
|      5|   675

In [76]:
coluna = 'CS25SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CS25SITE)|(count(CASE WHEN (isnan(CS25SITE) OR (CS25SITE IS NULL)) THEN CS25SITE END) AS `CS25SITE` / 10450709)|CS25SITE|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      17|                                                                                  0.42365240482727057| 4427468|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CS25SITE|  count|
+--------+-------+
|     988|5921117|
|    null|4427468|
|     981|  49577|
|     982|  23423|
|     100|  12931|
|       0|   4503|
|       2|   3615|
|      10|   3174|
|      40|   3041|
|      20|    771|
|       1|    482|
|      30|    128|
|      60|    123|
|    

In [77]:
coluna = 'DAJCCT'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+-------+
|count(DISTINCT DAJCCT)|(count(CASE WHEN (isnan(DAJCCT) OR (DAJCCT IS NULL)) THEN DAJCCT END) AS `DAJCCT` / 10450709)| DAJCCT|
+----------------------+---------------------------------------------------------------------------------------------+-------+
|                    34|                                                                            0.514264056151597|5374424|
+----------------------+---------------------------------------------------------------------------------------------+-------+

+------+-------+
|DAJCCT|  count|
+------+-------+
|  null|5374424|
|    88| 982448|
|    20| 558984|
|    18| 483553|
|    10| 462437|
|     5| 453614|
|    30| 408815|
|    40| 339077|
|    12| 270734|
|    15| 228435|
|    29| 156042|
|    23| 128759|
|     1| 101548|
|    21|  88946|
|    31|  77022|
|    22|  68238|
|    32|  51448|
|    19|  44641|
|    41

In [78]:
coluna = 'DAJCCN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+-------+
|count(DISTINCT DAJCCN)|(count(CASE WHEN (isnan(DAJCCN) OR (DAJCCN IS NULL)) THEN DAJCCN END) AS `DAJCCN` / 10450709)| DAJCCN|
+----------------------+---------------------------------------------------------------------------------------------+-------+
|                    22|                                                                           0.5070111511094606|5298626|
+----------------------+---------------------------------------------------------------------------------------------+-------+

+------+-------+
|DAJCCN|  count|
+------+-------+
|  null|5298626|
|     0|2979278|
|    88| 982917|
|    10| 368084|
|    20| 275766|
|     1| 181594|
|    11| 111332|
|    30|  75801|
|    21|  41172|
|    18|  29713|
|    22|  21696|
|    31|  16813|
|     2|  14370|
|    12|  12838|
|    23|  12287|
|    19|  11893|
|    29|   6696|
|    33|   3196|
|    32

In [79]:
coluna = 'DAJCCM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+-------+
|count(DISTINCT DAJCCM)|(count(CASE WHEN (isnan(DAJCCM) OR (DAJCCM IS NULL)) THEN DAJCCM END) AS `DAJCCM` / 10450709)| DAJCCM|
+----------------------+---------------------------------------------------------------------------------------------+-------+
|                     7|                                                                           0.4943855005435516|5166679|
+----------------------+---------------------------------------------------------------------------------------------+-------+

+------+-------+
|DAJCCM|  count|
+------+-------+
|  null|5166679|
|     0|3659048|
|    88| 982448|
|    10| 580937|
|    12|  37577|
|    13|  11518|
|    11|   6816|
|    19|   5686|
+------+-------+



In [80]:
coluna = 'DAJCCSTG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT DAJCCSTG)|(count(CASE WHEN (isnan(DAJCCSTG) OR (DAJCCSTG IS NULL)) THEN DAJCCSTG END) AS `DAJCCSTG` / 10450709)|DAJCCSTG|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      53|                                                                                   0.5091710045701205| 5321198|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|DAJCCSTG|  count|
+--------+-------+
|    null|5321198|
|      88| 757531|
|      10| 745555|
|      70| 646492|
|      30| 621919|
|       0| 440491|
|      32| 314103|
|      12| 247975|
|      53| 189290|
|      15| 166885|
|      50| 157699|
|      33| 157281|
|      52| 152036|
|    

In [81]:
coluna = 'DSS1977S'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT DSS1977S)|(count(CASE WHEN (isnan(DSS1977S) OR (DSS1977S IS NULL)) THEN DSS1977S END) AS `DSS1977S` / 10450709)|DSS1977S|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       8|                                                                                   0.5043942951621752| 5271278|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|DSS1977S|  count|
+--------+-------+
|    null|5271278|
|       1|2272564|
|       7|1250098|
|       0| 537873|
|       3| 380085|
|       2| 315914|
|       4| 200620|
|       8| 171293|
|       5|  50984|
+--------+-------+



In [82]:
coluna = 'SCSSM2KO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SCSSM2KO)|(count(CASE WHEN (isnan(SCSSM2KO) OR (SCSSM2KO IS NULL)) THEN SCSSM2KO END) AS `SCSSM2KO` / 10450709)|SCSSM2KO|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       8|                                                                                  0.45923955972747876| 4799379|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SCSSM2KO|  count|
+--------+-------+
|    null|4799379|
|       1|2435898|
|       7|1329125|
|       0| 590225|
|       3| 409071|
|       2| 400888|
|       4| 240082|
|       8| 188616|
|       5|  57425|
+--------+-------+



In [83]:
coluna = 'DAJCCFL'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT DAJCCFL)|(count(CASE WHEN (isnan(DAJCCFL) OR (DAJCCFL IS NULL)) THEN DAJCCFL END) AS `DAJCCFL` / 10450709)| DAJCCFL|
+-----------------------+-------------------------------------------------------------------------------------------------+--------+
|                      0|                                                                                              1.0|10450709|
+-----------------------+-------------------------------------------------------------------------------------------------+--------+

+-------+--------+
|DAJCCFL|   count|
+-------+--------+
|   null|10450709|
+-------+--------+



In [84]:
coluna = 'CSVFIRST'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CSVFIRST)|(count(CASE WHEN (isnan(CSVFIRST) OR (CSVFIRST IS NULL)) THEN CSVFIRST END) AS `CSVFIRST` / 10450709)|CSVFIRST|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      17|                                                                                   0.4711657362194278| 4924016|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CSVFIRST|  count|
+--------+-------+
|    null|4924016|
|   20550|1136906|
|   20440| 939021|
|   10401| 926082|
|   10200| 674167|
|   10300| 531837|
|   20302| 455449|
|   20200| 387476|
|   10100| 233896|
|   10400| 156071|
|   10002|  44824|
|   20100|  24028|
|   10004|   8627|
|   1

In [85]:
coluna = 'CSVLATES'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CSVLATES)|(count(CASE WHEN (isnan(CSVLATES) OR (CSVLATES IS NULL)) THEN CSVLATES END) AS `CSVLATES` / 10450709)|CSVLATES|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       1|                                                                                   0.4711657362194278| 4924016|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CSVLATES|  count|
+--------+-------+
|   20550|5526693|
|    null|4924016|
+--------+-------+



In [86]:
coluna = 'CSVCURRENT'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+--------------------------+-------------------------------------------------------------------------------------------------------------+----------+
|count(DISTINCT CSVCURRENT)|(count(CASE WHEN (isnan(CSVCURRENT) OR (CSVCURRENT IS NULL)) THEN CSVCURRENT END) AS `CSVCURRENT` / 10450709)|CSVCURRENT|
+--------------------------+-------------------------------------------------------------------------------------------------------------+----------+
|                         5|                                                                                           0.4711657362194278|   4924016|
+--------------------------+-------------------------------------------------------------------------------------------------------------+----------+

+----------+-------+
|CSVCURRENT|  count|
+----------+-------+
|      null|4924016|
|     20510|1913049|
|     20550|1566954|
|     20540| 958602|
|     20520| 652108|
|     20530| 435980|
+----------+-------+



In [87]:
coluna = 'SURGPRIF'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SURGPRIF)|(count(CASE WHEN (isnan(SURGPRIF) OR (SURGPRIF IS NULL)) THEN SURGPRIF END) AS `SURGPRIF` / 10450709)|SURGPRIF|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      73|                                                                                  0.25072145822833647| 2620217|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SURGPRIF|  count|
+--------+-------+
|       0|2659581|
|    null|2620217|
|      50| 764214|
|      30| 612294|
|      98| 536638|
|      22| 488736|
|      40| 402044|
|      27| 392917|
|      23| 182459|
|      51| 179313|
|      45| 158129|
|      20| 144046|
|      41| 143035|
|    

In [88]:
coluna = 'SURGSCOF'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SURGSCOF)|(count(CASE WHEN (isnan(SURGSCOF) OR (SURGSCOF IS NULL)) THEN SURGSCOF END) AS `SURGSCOF` / 10450709)|SURGSCOF|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       8|                                                                                   0.5848287422413159| 6111875|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SURGSCOF|  count|
+--------+-------+
|    null|6111875|
|       0|3068035|
|       5| 896046|
|       4| 210956|
|       2|  58890|
|       1|  56700|
|       3|  30074|
|       6|  13214|
|       7|   4919|
+--------+-------+



In [89]:
coluna = 'SURGSITF'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SURGSITF)|(count(CASE WHEN (isnan(SURGSITF) OR (SURGSITF IS NULL)) THEN SURGSITF END) AS `SURGSITF` / 10450709)|SURGSITF|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       6|                                                                                   0.3962980884837574| 4141596|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SURGSITF|  count|
+--------+-------+
|       0|6143339|
|    null|4141596|
|       4|  55210|
|       2|  52962|
|       1|  38313|
|       3|  11950|
|       5|   7339|
+--------+-------+



In [90]:
coluna = 'NUMNODES'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT NUMNODES)|(count(CASE WHEN (isnan(NUMNODES) OR (NUMNODES IS NULL)) THEN NUMNODES END) AS `NUMNODES` / 10450709)|NUMNODES|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      95|                                                                                   0.8649196910946425| 9039024|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|NUMNODES|  count|
+--------+-------+
|    null|9039024|
|       0| 912336|
|       1|  38548|
|       2|  37139|
|       3|  28336|
|       4|  25830|
|       5|  24271|
|       6|  23945|
|       7|  23076|
|       8|  22637|
|      10|  22622|
|       9|  22052|
|      11|  20605|
|    

In [135]:
coluna = 'NO_SURG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT NO_SURG)|(count(CASE WHEN (isnan(NO_SURG) OR (NO_SURG IS NULL)) THEN NO_SURG END) AS `NO_SURG` / 10450709)|NO_SURG|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                      7|                                                                             0.021188897327444483| 221439|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|NO_SURG|  count|
+-------+-------+
|      0|6196010|
|      1|3018833|
|      6| 718999|
|   null| 221439|
|      2| 154165|
|      7| 100173|
|      8|  34393|
|      5|   6697|
+-------+-------+



In [92]:
coluna = 'SS_SURG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT SS_SURG)|(count(CASE WHEN (isnan(SS_SURG) OR (SS_SURG IS NULL)) THEN SS_SURG END) AS `SS_SURG` / 10450709)|SS_SURG|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                     37|                                                                               0.8007713160896548|8368628|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|SS_SURG|  count|
+-------+-------+
|   null|8368628|
|     90| 423529|
|      0| 303532|
|     50| 230882|
|     40| 225857|
|      2| 198408|
|     10| 185386|
|     20| 157102|
|     30| 146052|
|     60|  76649|
|      1|  30801|
|     70|  21997|
|     80|  21492|
|     35|  11845|
|     58|   9451|
|      5|   869

In [93]:
coluna = 'SURGSCOP'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SURGSCOP)|(count(CASE WHEN (isnan(SURGSCOP) OR (SURGSCOP IS NULL)) THEN SURGSCOP END) AS `SURGSCOP` / 10450709)|SURGSCOP|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       7|                                                                                   0.8899885165685888| 9301011|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SURGSCOP|  count|
+--------+-------+
|    null|9301011|
|       0| 835044|
|       1| 256989|
|       2|  26693|
|       4|  14477|
|       3|  11921|
|       5|   3997|
|       6|    577|
+--------+-------+



In [94]:
coluna = 'SURGSITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SURGSITE)|(count(CASE WHEN (isnan(SURGSITE) OR (SURGSITE IS NULL)) THEN SURGSITE END) AS `SURGSITE` / 10450709)|SURGSITE|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       9|                                                                                    0.856474618133564| 8950767|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SURGSITE|  count|
+--------+-------+
|    null|8950767|
|       0|1458304|
|       2|  11741|
|       4|   9528|
|       1|   8908|
|       6|   5096|
|       3|   3900|
|       5|   1985|
|       7|    450|
|       8|     30|
+--------+-------+



In [95]:
coluna = 'REC_NO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+------+
|count(DISTINCT REC_NO)|(count(CASE WHEN (isnan(REC_NO) OR (REC_NO IS NULL)) THEN REC_NO END) AS `REC_NO` / 10450709)|REC_NO|
+----------------------+---------------------------------------------------------------------------------------------+------+
|                    47|                                                                                          0.0|     0|
+----------------------+---------------------------------------------------------------------------------------------+------+

+------+-------+
|REC_NO|  count|
+------+-------+
|     1|8951875|
|     2|1264423|
|     3| 190880|
|     4|  32415|
|     5|   7049|
|     6|   2187|
|     7|    819|
|     8|    399|
|     9|    223|
|    10|    140|
|    11|     77|
|    12|     42|
|    13|     32|
|    14|     23|
|    15|     18|
|    16|     14|
|    17|     13|
|    18|     13|
|    19|    

In [96]:
coluna = 'TYPE_FU'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT TYPE_FU)|(count(CASE WHEN (isnan(TYPE_FU) OR (TYPE_FU IS NULL)) THEN TYPE_FU END) AS `TYPE_FU` / 10450709)|TYPE_FU|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                      4|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+--------+
|TYPE_FU|   count|
+-------+--------+
|      2|10324061|
|      1|  119877|
|      4|    6770|
|      3|       1|
+-------+--------+



In [97]:
coluna = 'AGE_1REC'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT AGE_1REC)|(count(CASE WHEN (isnan(AGE_1REC) OR (AGE_1REC IS NULL)) THEN AGE_1REC END) AS `AGE_1REC` / 10450709)|AGE_1REC|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      19|                                                                                 1.351104504010206...|    1412|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|AGE_1REC|  count|
+--------+-------+
|      14|1419118|
|      15|1357677|
|      13|1272972|
|      16|1198878|
|      12|1075249|
|      17| 895414|
|      11| 834226|
|      18| 761816|
|      10| 580107|
|       9| 382069|
|       8| 237111|
|       7| 158058|
|       6| 101942|
|    

In [98]:
coluna = 'SITERWHO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SITERWHO)|(count(CASE WHEN (isnan(SITERWHO) OR (SITERWHO IS NULL)) THEN SITERWHO END) AS `SITERWHO` / 10450709)|SITERWHO|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      80|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SITERWHO|  count|
+--------+-------+
|   26000|1697178|
|   28010|1309913|
|   22030|1247844|
|   25010| 657812|
|   29010| 412260|
|   37000| 313259|
|   29020| 283578|
|   27020| 283551|
|   33041| 261495|
|   21100| 244562|
|   21048| 223888|
|   21052| 215270|
|   32010| 210323|
|   2

In [170]:
coluna = 'ICDOTO9V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT ICDOTO9V)|(count(CASE WHEN (isnan(ICDOTO9V) OR (ICDOTO9V IS NULL)) THEN ICDOTO9V END) AS `ICDOTO9V` / 10450709)|ICDOTO9V|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                     403|                                                                                 0.028417210736611267|  296980|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|ICDOTO9V|  count|
+--------+-------+
|     185|1309244|
|    1623| 591517|
|    1744| 460445|
|    null| 296980|
|    2330| 292270|
|    1748| 289728|
|    1625| 286788|
|    1820| 281042|
|    1890| 261060|
|    1749| 243211|
|    1629| 239417|
|     193| 210252|
|    1533| 207903|
|    

In [172]:
df = df.replace('9999', None, subset='ICDOT10V')

In [173]:
coluna = 'ICDOT10V'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT ICDOT10V)|(count(CASE WHEN (isnan(ICDOT10V) OR (ICDOT10V IS NULL)) THEN ICDOT10V END) AS `ICDOT10V` / 10450709)|ICDOT10V|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                     481|                                                                                 0.028423813159470807|  297049|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|ICDOT10V|  count|
+--------+-------+
|     C61|1309263|
|    C341| 591410|
|    C504| 462442|
|    null| 297049|
|    C508| 291666|
|    C343| 286662|
|     C64| 259538|
|    C509| 243695|
|    C349| 239036|
|    C541| 218396|
|    D059| 217941|
|     C73| 210252|
|    C187| 207941|
|    

In [101]:
coluna = 'ICCC3WHO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT ICCC3WHO)|(count(CASE WHEN (isnan(ICCC3WHO) OR (ICCC3WHO IS NULL)) THEN ICCC3WHO END) AS `ICCC3WHO` / 10450709)|ICCC3WHO|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      48|                                                                                  0.07400923707664236|  773449|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|ICCC3WHO|  count|
+--------+-------+
|     116|6693775|
|    null| 773449|
|      22| 484048|
|     114| 412610|
|      62| 245413|
|     122| 220151|
|     112| 208834|
|     104| 157718|
|      35| 149752|
|      72| 142691|
|      11| 139110|
|      32|  94890|
|      12|  81913|
|    

In [102]:
coluna = 'ICCC3XWHO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-------------------------+---------------------------------------------------------------------------------------------------------+---------+
|count(DISTINCT ICCC3XWHO)|(count(CASE WHEN (isnan(ICCC3XWHO) OR (ICCC3XWHO IS NULL)) THEN ICCC3XWHO END) AS `ICCC3XWHO` / 10450709)|ICCC3XWHO|
+-------------------------+---------------------------------------------------------------------------------------------------------+---------+
|                      114|                                                                                      0.07400923707664236|   773449|
+-------------------------+---------------------------------------------------------------------------------------------------------+---------+

+---------+-------+
|ICCC3XWHO|  count|
+---------+-------+
|      106|2452954|
|      102|1384511|
|      100|1167469|
|       98| 990123|
|     null| 773449|
|       11| 415889|
|       95| 412610|
|      104| 406323|
|       40| 245413|
|      114| 220151|
|       93| 208834|

In [103]:
coluna = 'BEHTREND'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT BEHTREND)|(count(CASE WHEN (isnan(BEHTREND) OR (BEHTREND IS NULL)) THEN BEHTREND END) AS `BEHTREND` / 10450709)|BEHTREND|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       7|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|BEHTREND|  count|
+--------+-------+
|       3|9428053|
|       2| 718109|
|       0| 172094|
|       4| 102550|
|       1|  16522|
|       6|   7081|
|       5|   6300|
+--------+-------+



In [104]:
coluna = 'HISTREC'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT HISTREC)|(count(CASE WHEN (isnan(HISTREC) OR (HISTREC IS NULL)) THEN HISTREC END) AS `HISTREC` / 10450709)|HISTREC|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                     57|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|HISTREC|  count|
+-------+-------+
|      5|4056161|
|      9|1543898|
|      2| 853021|
|      1| 710140|
|     15| 676221|
|      4| 436032|
|      8| 309092|
|     42| 283400|
|      0| 249535|
|     50| 130363|
|     35| 128299|
|     45| 125349|
|     51| 114250|
|     37| 100481|
|     55|  62872|
|     25|  6218

In [105]:
coluna = 'HISTRECB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT HISTRECB)|(count(CASE WHEN (isnan(HISTRECB) OR (HISTRECB IS NULL)) THEN HISTRECB END) AS `HISTRECB` / 10450709)|HISTRECB|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      29|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+--------+
|HISTRECB|   count|
+--------+--------+
|      98|10114749|
|      19|  100448|
|       3|   62956|
|      25|   47394|
|      18|   23663|
|      11|   13873|
|      27|   11005|
|      22|   10119|
|      12|    9349|
|       2|    8725|
|       6|    5837|
|       4|    5800|
|      21

In [106]:
coluna = 'CS0204SCHEMA'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------------+---------------------------------------------------------------------------------------------------------------------+------------+
|count(DISTINCT CS0204SCHEMA)|(count(CASE WHEN (isnan(CS0204SCHEMA) OR (CS0204SCHEMA IS NULL)) THEN CS0204SCHEMA END) AS `CS0204SCHEMA` / 10450709)|CS0204SCHEMA|
+----------------------------+---------------------------------------------------------------------------------------------------------------------+------------+
|                         151|                                                                                                                  0.0|           0|
+----------------------------+---------------------------------------------------------------------------------------------------------------------+------------+

+------------+-------+
|CS0204SCHEMA|  count|
+------------+-------+
|          13|1697178|
|         129|1309913|
|          63|1248521|
|          18| 722455|
|          95| 659706|
|    

In [107]:
coluna = 'RAC_RECA'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT RAC_RECA)|(count(CASE WHEN (isnan(RAC_RECA) OR (RAC_RECA IS NULL)) THEN RAC_RECA END) AS `RAC_RECA` / 10450709)|RAC_RECA|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       3|                                                                                 0.010696116407030375|  111782|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|RAC_RECA|  count|
+--------+-------+
|       1|8661749|
|       2|1018928|
|       3| 658250|
|    null| 111782|
+--------+-------+



In [108]:
coluna = 'RAC_RECY'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT RAC_RECY)|(count(CASE WHEN (isnan(RAC_RECY) OR (RAC_RECY IS NULL)) THEN RAC_RECY END) AS `RAC_RECY` / 10450709)|RAC_RECY|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       4|                                                                                 0.010696116407030375|  111782|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|RAC_RECY|  count|
+--------+-------+
|       1|8661749|
|       2|1018928|
|       4| 606592|
|    null| 111782|
|       3|  51658|
+--------+-------+



In [109]:
coluna = 'ORIGRECB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT ORIGRECB)|(count(CASE WHEN (isnan(ORIGRECB) OR (ORIGRECB IS NULL)) THEN ORIGRECB END) AS `ORIGRECB` / 10450709)|ORIGRECB|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       2|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|ORIGRECB|  count|
+--------+-------+
|       0|9604368|
|       1| 846341|
+--------+-------+



In [110]:
coluna = 'HST_STGA'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT HST_STGA)|(count(CASE WHEN (isnan(HST_STGA) OR (HST_STGA IS NULL)) THEN HST_STGA END) AS `HST_STGA` / 10450709)|HST_STGA|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       5|                                                                                  0.23876073862548464| 2495219|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|HST_STGA|  count|
+--------+-------+
|       1|2937870|
|    null|2495219|
|       4|1765716|
|       2|1675967|
|       8| 908943|
|       0| 666994|
+--------+-------+



In [111]:
coluna = 'AJCC_STG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT AJCC_STG)|(count(CASE WHEN (isnan(AJCC_STG) OR (AJCC_STG IS NULL)) THEN AJCC_STG END) AS `AJCC_STG` / 10450709)|AJCC_STG|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      24|                                                                                   0.6787382559403385| 7093296|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|AJCC_STG|  count|
+--------+-------+
|    null|7093296|
|      88|1168618|
|      90| 580573|
|      10| 333697|
|      40| 296730|
|      20| 158085|
|       0| 144141|
|      30| 129588|
|      21| 110002|
|      32|  91109|
|      18|  79914|
|      98|  59560|
|      22|  59238|
|    

In [112]:
coluna = 'AJ_3SEER'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT AJ_3SEER)|(count(CASE WHEN (isnan(AJ_3SEER) OR (AJ_3SEER IS NULL)) THEN AJ_3SEER END) AS `AJ_3SEER` / 10450709)|AJ_3SEER|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      23|                                                                                   0.6786398894084602| 7092268|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|AJ_3SEER|  count|
+--------+-------+
|    null|7092268|
|      88|1168618|
|      10| 520418|
|      90| 329805|
|      40| 306743|
|      20| 182105|
|       0| 148808|
|      30| 142663|
|      21| 115483|
|      18|  98002|
|      32|  96399|
|      22|  61800|
|      31|  58522|
|    

In [113]:
coluna = 'SSS77VZ'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT SSS77VZ)|(count(CASE WHEN (isnan(SSS77VZ) OR (SSS77VZ IS NULL)) THEN SSS77VZ END) AS `SSS77VZ` / 10450709)|SSS77VZ|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                      8|                                                                               0.8928346392574896|9330755|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|SSS77VZ|  count|
+-------+-------+
|   null|9330755|
|      1| 410336|
|      7| 257324|
|      8| 155203|
|      0|  80836|
|      3|  79909|
|      2|  74768|
|      4|  49605|
|      5|  11973|
+-------+-------+



In [114]:
coluna = 'SSSM2KPZ'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT SSSM2KPZ)|(count(CASE WHEN (isnan(SSSM2KPZ) OR (SSSM2KPZ IS NULL)) THEN SSSM2KPZ END) AS `SSSM2KPZ` / 10450709)|SSSM2KPZ|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       7|                                                                                   0.8953560949788192| 9357106|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|SSSM2KPZ|  count|
+--------+-------+
|    null|9357106|
|       1| 496319|
|       7| 257823|
|       0| 107091|
|       2|  89896|
|       3|  80161|
|       4|  51086|
|       5|  11227|
+--------+-------+



In [115]:
coluna = 'FIRSTPRM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT FIRSTPRM)|(count(CASE WHEN (isnan(FIRSTPRM) OR (FIRSTPRM IS NULL)) THEN FIRSTPRM END) AS `FIRSTPRM` / 10450709)|FIRSTPRM|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       2|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|FIRSTPRM|  count|
+--------+-------+
|       1|7977634|
|       0|2473075|
+--------+-------+



In [116]:
coluna = 'ST_CNTY'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT ST_CNTY)|(count(CASE WHEN (isnan(ST_CNTY) OR (ST_CNTY IS NULL)) THEN ST_CNTY END) AS `ST_CNTY` / 10450709)|ST_CNTY|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                    626|                                                                                              0.0|      0|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|ST_CNTY|  count|
+-------+-------+
|   6037|1004075|
|  26163| 459760|
|  53033| 336309|
|   6073| 257771|
|  26125| 249811|
|   6001| 244549|
|   6059| 242334|
|   9003| 203161|
|   9001| 201989|
|   9009| 196631|
|   6085| 185563|
|   6013| 178706|
|   6075| 178227|
|  26099| 174267|
|   6065| 161536|
|   6081| 14186

In [117]:
coluna = 'CODPUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+------+
|count(DISTINCT CODPUB)|(count(CASE WHEN (isnan(CODPUB) OR (CODPUB IS NULL)) THEN CODPUB END) AS `CODPUB` / 10450709)|CODPUB|
+----------------------+---------------------------------------------------------------------------------------------+------+
|                    94|                                                                                          0.0|     0|
+----------------------+---------------------------------------------------------------------------------------------+------+

+------+-------+
|CODPUB|  count|
+------+-------+
|     0|4590865|
| 22030|1031839|
| 50060| 683873|
| 21040| 333996|
| 50300| 322710|
| 26000| 282651|
| 37000| 282503|
| 21100| 239538|
| 28010| 186596|
| 50130| 151824|
| 33040| 145363|
| 50080| 144770|
| 21020| 104855|
| 29010| 104293|
| 27040|  99022|
| 21071|  96260|
| 31010|  95249|
| 21010|  91819|
| 29020|  84

In [118]:
coluna = 'CODPUBKM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT CODPUBKM)|(count(CASE WHEN (isnan(CODPUBKM) OR (CODPUBKM IS NULL)) THEN CODPUBKM END) AS `CODPUBKM` / 10450709)|CODPUBKM|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                      96|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|CODPUBKM|  count|
+--------+-------+
|       0|4590865|
|   22030|1031839|
|   50060| 683873|
|   21040| 333996|
|   50300| 322710|
|   26000| 282651|
|   37000| 271243|
|   21100| 239538|
|   28010| 186596|
|   50130| 151824|
|   33040| 145363|
|   50080| 144770|
|   21020| 104855|
|   2

In [119]:
coluna = 'STAT_REC'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|count(DISTINCT STAT_REC)|(count(CASE WHEN (isnan(STAT_REC) OR (STAT_REC IS NULL)) THEN STAT_REC END) AS `STAT_REC` / 10450709)|STAT_REC|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+
|                       2|                                                                                                  0.0|       0|
+------------------------+-----------------------------------------------------------------------------------------------------+--------+

+--------+-------+
|STAT_REC|  count|
+--------+-------+
|       0|5859844|
|       1|4590865|
+--------+-------+



In [120]:
coluna = 'IHSLINK'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|count(DISTINCT IHSLINK)|(count(CASE WHEN (isnan(IHSLINK) OR (IHSLINK IS NULL)) THEN IHSLINK END) AS `IHSLINK` / 10450709)|IHSLINK|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+
|                      2|                                                                              0.07771424886101029| 812169|
+-----------------------+-------------------------------------------------------------------------------------------------+-------+

+-------+-------+
|IHSLINK|  count|
+-------+-------+
|      0|9612483|
|   null| 812169|
|      1|  26057|
+-------+-------+



In [121]:
coluna = 'SUMM2K'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+----------------------+---------------------------------------------------------------------------------------------+-------+
|count(DISTINCT SUMM2K)|(count(CASE WHEN (isnan(SUMM2K) OR (SUMM2K IS NULL)) THEN SUMM2K END) AS `SUMM2K` / 10450709)| SUMM2K|
+----------------------+---------------------------------------------------------------------------------------------+-------+
|                     5|                                                                           0.3664323635841358|3829478|
+----------------------+---------------------------------------------------------------------------------------------+-------+

+------+-------+
|SUMM2K|  count|
+------+-------+
|  null|3829478|
|     1|3146620|
|     2|1389898|
|     7|1345120|
|     0| 739592|
|     8|      1|
+------+-------+



In [122]:
coluna = 'AYASITERWHO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

+---------------------------+-----------------------------------------------------------------------------------------------------------------+-----------+
|count(DISTINCT AYASITERWHO)|(count(CASE WHEN (isnan(AYASITERWHO) OR (AYASITERWHO IS NULL)) THEN AYASITERWHO END) AS `AYASITERWHO` / 10450709)|AYASITERWHO|
+---------------------------+-----------------------------------------------------------------------------------------------------------------+-----------+
|                         56|                                                                                              0.07648284915406219|     799300|
+---------------------------+-----------------------------------------------------------------------------------------------------------------+-----------+

+-----------+-------+
|AYASITERWHO|  count|
+-----------+-------+
|         36|1384686|
|         41|1373396|
|         35|1168828|
|         42|1004858|
|       null| 799300|
|         29| 412608|
|         38| 406335|

In [123]:
coluna = 'LYMSUBRWHO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

KeyboardInterrupt: 

In [ ]:
coluna = 'VSRTSADX'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ODTHCLASS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CSTSEVAL'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CSRGEVAL'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CSMTEVAL'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'INTPRIM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ERSTATUS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'PRSTATUS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CSSCHEMA'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS8SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS10SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS11SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS13SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS15SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS16SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'VASINV'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'SRV_TIME_MON'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'SRV_TIME_MON_FLAG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'INSREC_PUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DAJCC7T'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DAJCC7N'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DAJCC7M'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DAJCC7STG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ADJTM_6VALUE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ADJNM_6VALUE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ADJM_6VALUE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ADJAJCCSTG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS7SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS9SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CS12SITE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'HER2'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'BRST_SUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'ANNARBOR'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'SCMETSDXB_PUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'SCMETSDXBR_PUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'SCMETSDXLIV_PUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'SCMETSDXLUNG_PUB'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'T_VALUE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'N_VALUE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'M_VALUE'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'MALIGCOUNT'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'BENBORDCOUNT'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'TUMSIZS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DSRPSG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCT'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCTS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCNS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'DASRCMS'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'TNMEDNUM'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'METSDXLN'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'METSDXO'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'RADIATNR'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'RAD_BRNR'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'RAD_SURG'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()

In [ ]:
coluna = 'CHEMO_RX_REC'
df.select([F.countDistinct(coluna), F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna) / TAM_TOTAL, F.count(F.when(F.isnan(coluna) | F.col(coluna).isNull(), coluna)).alias(coluna)]).show()
df.groupby(coluna).count().orderBy('count', ascending=False).show()